In [ ]:
!pip install streamlit langchain-groq chromadb langchain-community langchain-text-splitters sentence-transformers pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 

In [ ]:
import os

# Create folders
os.makedirs('project_3_app/company_docs', exist_ok=True)

# Create company documents
with open('project_3_app/company_docs/hr_policy.txt', 'w') as f:
    f.write("""Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.""")

with open('project_3_app/company_docs/benefits.txt', 'w') as f:
    f.write("""Employee Benefits Guide

Health Insurance: All full-time employees receive comprehensive health insurance coverage including medical, dental, and vision. Coverage begins on the first day of employment.

401k Plan: Employees are eligible for the company 401k plan after 90 days. The company matches up to 5% of employee contributions.

Gym Membership: All employees receive a free gym membership as part of the wellness program.""")

with open('project_3_app/company_docs/it_policy.txt', 'w') as f:
    f.write("""IT Security Policy

Password Policy: All employees must use strong passwords with at least 12 characters. Passwords must be changed every 90 days.

Software Installation: Employees are not allowed to install unauthorized software on company devices. All software must be approved by the IT department.

Data Backup: All important data must be backed up to the company cloud storage daily. Personal storage devices are not allowed for company data.""")

print("Project structure created!")

Project structure created!


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Custom embedding function
class SentenceTransformerEF(chromadb.EmbeddingFunction):
    def __call__(self, input):
        return [embedding_model.encode(text).tolist() for text in input]

st_ef = SentenceTransformerEF()

# Create ChromaDB
chroma_client = chromadb.PersistentClient(path='./project_3_app/chroma_db')

# Create collection
collection = chroma_client.get_or_create_collection(
    name='company_docs',
    embedding_function=st_ef
)

# Load and index documents
loader = DirectoryLoader('project_3_app/company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

if collection.count() == 0:
    collection.add(
        documents=[chunk.page_content for chunk in chunks],
        ids=[f'doc_{i}' for i in range(len(chunks))],
        metadatas=[{'source': chunk.metadata.get('source', 'unknown')} for chunk in chunks]
    )
    print(f"Added {len(chunks)} chunks to ChromaDB")
else:
    print(f"Collection already has {collection.count()} documents")

/tmp/ipykernel_11323/533047533.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_11323/533047533.py:14: DeprecationWarning: The class SentenceTransformerEF does not implement __init__. This will be required in a future version.
  st_ef = SentenceTransformerEF()


Added 3 chunks to ChromaDB


In [ ]:
app_code = '''
import streamlit as st
import os
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq

# Page config
st.set_page_config(
    page_title="AI Assistant",
    page_icon="🤖",
    layout="wide"
)

# Embedding function
class SentenceTransformerEF(chromadb.EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
    def __call__(self, input):
        return [self.model.encode(text).tolist() for text in input]

@st.cache_resource
def init_chromadb():
    client = chromadb.PersistentClient(path="./project_3_app/chroma_db")
    collection = client.get_collection(
        name="company_docs",
        embedding_function=SentenceTransformerEF()
    )
    return collection

@st.cache_resource
def init_llm():
    return ChatGroq(
        api_key="your-groq-api-key-here",
        model="llama-3.1-8b-instant",
        temperature=0
    )

collection = init_chromadb()
llm = init_llm()

def get_rag_response(query, n_results=3):
    try:
        results = collection.query(query_texts=[query], n_results=n_results)
        if not results["documents"][0]:
            return "No relevant information found in documents."
        context = "\\n\\n---\\n\\n".join(results["documents"][0])
        prompt = f"""You are a helpful HR assistant. Answer using ONLY the context below.
If not in context, say so. Be concise and friendly.

Context:
{context}

Question: {query}

Answer:"""
        messages = [{"role": "user", "content": prompt}]
        response = llm.invoke(messages)
        return response.content
    except Exception as e:
        return f"Error: {str(e)}"

# Sidebar
with st.sidebar:
    st.header("About")
    st.markdown("""
This AI assistant can answer questions about:
- Vacation policies
- Remote work guidelines
- Parental leave
- Benefits information

Powered by:
- Groq LLaMA 3.1
- ChromaDB vector search
- Semantic RAG
    """)
    st.divider()
    st.metric("Documents Indexed", collection.count())
    st.metric("Messages in Chat", len(st.session_state.get("messages", [])))
    st.divider()
    if st.button("Clear Chat History"):
        st.session_state.messages = []
        st.rerun()

# Title
st.title("🤖 Company Knowledge Assistant")
st.markdown("Ask me anything about company policies!")

# Initialize session state
if "messages" not in st.session_state:
    st.session_state.messages = []

# Welcome message
if len(st.session_state.messages) == 0:
    welcome = """👋 Hi! I am your company knowledge assistant. I can help you find information about:
- Vacation and time off policies
- Remote work guidelines
- Parental leave benefits
- And more!

Just ask me a question to get started."""
    with st.chat_message("assistant"):
        st.write(welcome)

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

# Chat input
if prompt := st.chat_input("Ask a question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Searching documents..."):
            response = get_rag_response(prompt)
        st.write(response)
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

with open('project_3_app/app.py', 'w') as f:
    f.write(app_code)

print("app.py created!")

app.py created!


In [ ]:
# Read the file
with open('project_3_app/app.py', 'r') as f:
    content = f.read()

# Replace with your actual API key
content = content.replace('your-groq-api-key-here', 'your-groq-api-key-here')

# Save it back
with open('project_3_app/app.py', 'w') as f:
    f.write(content)

print("API key updated!")

API key updated!


In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 4s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [ ]:
import subprocess
import threading
import time

# Start Streamlit in background
def run_streamlit():
    subprocess.run(['streamlit', 'run', 'project_3_app/app.py',
                   '--server.port', '8501',
                   '--server.headless', 'true'])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

# Wait for Streamlit to start
time.sleep(5)
print("Streamlit started!")

# Start localtunnel
tunnel = subprocess.Popen(['lt', '--port', '8501'], stdout=subprocess.PIPE)
time.sleep(3)
url = tunnel.stdout.readline().decode('utf-8').strip()
print(f"🚀 Your app is live at: {url}")

Streamlit started!
🚀 Your app is live at: your url is: https://sweet-weeks-watch.loca.lt


In [1]:
import subprocess
import threading
import time

# Restart Streamlit
def run_streamlit():
    subprocess.run(['streamlit', 'run', 'project_3_app/app.py',
                   '--server.port', '8501',
                   '--server.headless', 'true'])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(5)
print("Streamlit started!")

# Start fresh localtunnel
tunnel = subprocess.Popen(['lt', '--port', '8501'], stdout=subprocess.PIPE)
time.sleep(3)
url = tunnel.stdout.readline().decode('utf-8').strip()
print(f"🚀 Your app is live at: {url}")

Exception in thread Thread-3 (run_streamlit):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_2331/2888650517.py", line 7, in run_streamlit
  File "/usr/lib/python3.12/subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "/usr/lib/python3.12/subprocess.py", line 1955, in _execute_child
    raise child_exception_type(errno_num, err_msg, err_filename)
FileNotFoundError: [Errno 2] No such file or directory: 'streamlit'


Streamlit started!


FileNotFoundError: [Errno 2] No such file or directory: 'lt'

In [2]:
# Reinstall localtunnel
!npm install -g localtunnel

# Reinstall packages
!pip install streamlit langchain-groq chromadb langchain-community langchain-text-splitters sentence-transformers -q

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 11.16.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.16.0
npm notice To update run: npm install -g npm@11.16.0
npm notice
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [3]:
import os, subprocess, threading, time

# Recreate folders and docs
os.makedirs('project_3_app/company_docs', exist_ok=True)

with open('project_3_app/company_docs/hr_policy.txt', 'w') as f:
    f.write("""Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.""")

with open('project_3_app/company_docs/benefits.txt', 'w') as f:
    f.write("""Employee Benefits Guide

Health Insurance: All full-time employees receive comprehensive health insurance coverage including medical, dental, and vision. Coverage begins on the first day of employment.

401k Plan: Employees are eligible for the company 401k plan after 90 days. The company matches up to 5% of employee contributions.

Gym Membership: All employees receive a free gym membership as part of the wellness program.""")

with open('project_3_app/company_docs/it_policy.txt', 'w') as f:
    f.write("""IT Security Policy

Password Policy: All employees must use strong passwords with at least 12 characters. Passwords must be changed every 90 days.

Software Installation: Employees are not allowed to install unauthorized software on company devices. All software must be approved by the IT department.

Data Backup: All important data must be backed up to the company cloud storage daily. Personal storage devices are not allowed for company data.""")

print("Docs created!")

Docs created!


In [4]:
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

class SentenceTransformerEF(chromadb.EmbeddingFunction):
    def __call__(self, input):
        return [embedding_model.encode(text).tolist() for text in input]

st_ef = SentenceTransformerEF()

chroma_client = chromadb.PersistentClient(path='./project_3_app/chroma_db')

try:
    chroma_client.delete_collection('company_docs')
except:
    pass

collection = chroma_client.get_or_create_collection(
    name='company_docs',
    embedding_function=st_ef
)

loader = DirectoryLoader('project_3_app/company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

collection.add(
    documents=[chunk.page_content for chunk in chunks],
    ids=[f'doc_{i}' for i in range(len(chunks))],
    metadatas=[{'source': chunk.metadata.get('source', 'unknown')} for chunk in chunks]
)

print(f"ChromaDB ready with {collection.count()} chunks!")

/tmp/ipykernel_2331/2287470292.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2331/2287470292.py:12: DeprecationWarning: The class SentenceTransformerEF does not implement __init__. This will be required in a future version.
  st_ef = SentenceTransformerEF()


ChromaDB ready with 3 chunks!


In [5]:
app_code = '''
import streamlit as st
import os
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq

st.set_page_config(page_title="AI Assistant", page_icon="🤖", layout="wide")

class SentenceTransformerEF(chromadb.EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
    def __call__(self, input):
        return [self.model.encode(text).tolist() for text in input]

@st.cache_resource
def init_chromadb():
    client = chromadb.PersistentClient(path="./project_3_app/chroma_db")
    collection = client.get_collection(name="company_docs", embedding_function=SentenceTransformerEF())
    return collection

@st.cache_resource
def init_llm():
    return ChatGroq(api_key="YOUR_GROQ_API_KEY", model="llama-3.1-8b-instant", temperature=0)

collection = init_chromadb()
llm = init_llm()

def get_rag_response(query):
    try:
        results = collection.query(query_texts=[query], n_results=3)
        if not results["documents"][0]:
            return "No relevant information found."
        context = "\\n\\n---\\n\\n".join(results["documents"][0])
        prompt = f"""You are a helpful HR assistant. Answer using ONLY the context below. Be concise and friendly.

Context:
{context}

Question: {query}
Answer:"""
        response = llm.invoke([{"role": "user", "content": prompt}])
        return response.content
    except Exception as e:
        return f"Error: {str(e)}"

with st.sidebar:
    st.header("About")
    st.markdown("""
This AI assistant answers questions about:
- Vacation policies
- Remote work guidelines
- Parental leave
- Benefits information

Powered by:
- Groq LLaMA 3.1
- ChromaDB
- Semantic RAG
    """)
    st.divider()
    st.metric("Documents Indexed", collection.count())
    st.metric("Messages in Chat", len(st.session_state.get("messages", [])))
    st.divider()
    if st.button("Clear Chat History"):
        st.session_state.messages = []
        st.rerun()

st.title("🤖 Company Knowledge Assistant")
st.markdown("Ask me anything about company policies!")

if "messages" not in st.session_state:
    st.session_state.messages = []

if len(st.session_state.messages) == 0:
    with st.chat_message("assistant"):
        st.write("👋 Hi! I am your company knowledge assistant. Ask me about vacation, remote work, benefits, or IT policies!")

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

if prompt := st.chat_input("Ask a question..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    with st.chat_message("assistant"):
        with st.spinner("Searching documents..."):
            response = get_rag_response(prompt)
        st.write(response)
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

# Replace with your actual API key
app_code = app_code.replace('YOUR_GROQ_API_KEY', 'paste-your-groq-api-key-here')

with open('project_3_app/app.py', 'w') as f:
    f.write(app_code)

print("app.py created!")

app.py created!


In [6]:
import subprocess, threading, time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'project_3_app/app.py',
                   '--server.port', '8501',
                   '--server.headless', 'true'])

thread = threading.Thread(target=run_streamlit)
thread.daemon = True
thread.start()

time.sleep(5)
print("Streamlit started!")

tunnel = subprocess.Popen(['lt', '--port', '8501'], stdout=subprocess.PIPE)
time.sleep(3)
url = tunnel.stdout.readline().decode('utf-8').strip()
print(f"🚀 Your app is live at: {url}")

Streamlit started!
🚀 Your app is live at: your url is: https://wicked-views-sink.loca.lt
